In [1]:
# Import libraries we need and generate data
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Generate a synthetic dataset
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the shapes
print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (800, 20)
Testing set size: (200, 20)


In [2]:
class SimpleNN:
    def __init__(self, input_size, hidden_size, output_size):
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))

    def sigmoid(self, Z):
        return 1 / (1 + np.exp(-Z))

    def sigmoid_derivative(self, Z):
        return Z * (1 - Z)

    def softmax(self, Z):
        expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return expZ / np.sum(expZ, axis=1, keepdims=True)

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.sigmoid(self.Z1)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.softmax(self.Z2)
        return self.A2

    def compute_loss(self, Y):
        m = Y.shape[0]
        log_likelihood = -np.log(self.A2[range(m), Y])
        loss = np.sum(log_likelihood) / m
        return loss

    def backward(self, X, Y, learning_rate):
        m = X.shape[0]
        Y_one_hot = np.zeros((m, 2))
        Y_one_hot[np.arange(m), Y] = 1

        dZ2 = self.A2 - Y_one_hot
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.sigmoid_derivative(self.A1)
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2

    def train(self, X, Y, epochs, learning_rate):
        for epoch in range(epochs):
            self.forward(X)
            loss = self.compute_loss(Y)
            self.backward(X, Y, learning_rate)
            if epoch % 100 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

In [4]:
# Initialize and train the model
input_size = X_train.shape[1]
hidden_size = 64
output_size = 2
learning_rate = 0.1
epochs = 1000

model = SimpleNN(input_size, hidden_size, output_size)
model.train(X_train, y_train, epochs, learning_rate)

Epoch 0, Loss: 0.6932
Epoch 100, Loss: 0.6460
Epoch 200, Loss: 0.3915
Epoch 300, Loss: 0.3289
Epoch 400, Loss: 0.3181
Epoch 500, Loss: 0.3153
Epoch 600, Loss: 0.3144
Epoch 700, Loss: 0.3139
Epoch 800, Loss: 0.3136
Epoch 900, Loss: 0.3134


In [5]:
# Evaluate the model
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

train_acc = accuracy(y_train, model.predict(X_train))
test_acc = accuracy(y_test, model.predict(X_test))

print(f"Training Accuracy: {train_acc * 100:.2f}%")
print(f"Testing Accuracy: {test_acc * 100:.2f}%")

Training Accuracy: 88.25%
Testing Accuracy: 86.50%
